In [ ]:
import arcpy
from arcgis.gis import GIS
import pandas as pd
import numpy as np
import zipfile
import io
import requests
import os
cwd = os.getcwd()

In [ ]:
# creating ArcGIS workspace
arcpy.management.CreateFileGDB(
    out_folder_path=os.getcwd(),
    out_name="industrial-pollution-risk.gdb"
)

In [ ]:
# setting workspace
arcpy.env.workspace = os.getcwd() + "\\industrial-pollution-risk.gdb"
arcpy.env.overwriteOutput = True

## Public schools data (NCES)
The National Center for Education Statistics publishes a [yearly dataset](https://nces.ed.gov/programs/edge/Geographic/SchoolLocations) of all public schools in the US. 

In [ ]:
# downloading, uncompressing, and saving public schools data
url = "https://nces.ed.gov/programs/edge/data/EDGE_GEOCODE_PUBLICSCH_2425.zip"
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    zip_ref.extractall("EDGE_GEOCODE_PUBLICSCH_2425")
    
zip_file = "EDGE_GEOCODE_PUBLICSCH_2425/Shapefile_SCH.zip"
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("EDGE_GEOCODE_PUBLICSCH_2425")

In [ ]:
# importing public school data as a feature
arcpy.management.XYTableToPoint(
    in_table="EDGE_GEOCODE_PUBLICSCH_2425/EDGE_GEOCODE_PUBLICSCH_2425.dbf", 
    out_feature_class="SCH_XY", 
    x_field="LON", 
    y_field="LAT", 
    z_field=None, 
    coordinate_system=arcpy.SpatialReference(4326)
)

## EPA polluters data (TRI)
The EPA publishes a yearly dataset of industrial polluters called the [Toxics Release Inventory (TRI)](https://www.epa.gov/toxics-release-inventory-tri-program), which lists all industrial sites and their reported emissions. The TRI dataset splits emissions by chemical and includes descriptive information on each site.
The coordinates provided in the TRI dataset are in some cases estimated, so I'm also going to pull in geographic data from the EPA [Facility Registry Service](https://www.epa.gov/frs), which is a more robust dataset.

In [ ]:
# downloading & cleaning TRI data from EPA
url = "https://data.epa.gov/efservice/downloads/tri/mv_tri_basic_download/2022_US/csv"
tri = pd.read_csv(url)

tri = tri[[
    "2. TRIFD",
    "3. FRS ID",
    "4. FACILITY NAME",
    "6. CITY",
    "8. ST",
    "17. STANDARD PARENT CO NAME",
    "23. INDUSTRY SECTOR", 
    "37. CHEMICAL",
    "50. UNIT OF MEASURE",
    "65. ON-SITE RELEASE TOTAL"
]]

# converting `Dioxin and dioxin-like compounds` (in grams) to lbs
# this is not usually how these chemicals are analyzed (typically they are in very small amounts and are converted to 
# EPA-calculated Toxic Equivalency values to reflect their harmfulness) 
# but for this exercise they need to be in the same units as the other chemicals
tri.loc[tri["50. UNIT OF MEASURE"] == "Grams", "65. ON-SITE RELEASE TOTAL"] /= 453.59237
del tri["50. UNIT OF MEASURE"]

# group by facility and sum lbs
tri = tri.groupby([
    "2. TRIFD", 
    "3. FRS ID",
    "4. FACILITY NAME", 
    "6. CITY",
    "8. ST",
    "17. STANDARD PARENT CO NAME", 
    "23. INDUSTRY SECTOR"
    ])["65. ON-SITE RELEASE TOTAL"].sum()

# cleaning
tri = tri.reset_index()
tri["3. FRS ID"] = tri["3. FRS ID"].astype(int).astype(str)
tri.to_csv("TRI_summed.csv")

In [ ]:
# importing polluter data as a table
arcpy.conversion.ExportTable(
    in_table="TRI_summed.csv", 
    out_table="TRI_ET"
)

# cleaning FRS field for joining with FRS lat/lon
arcpy.management.CalculateField(
    in_table="TRI_ET",
    field="FRS_ID_str",
    expression='"!FRS_ID!".zfill(12)',
    expression_type="PYTHON3",
    code_block="",
    field_type="TEXT",
    enforce_domains="NO_ENFORCE_DOMAINS"
)

In [ ]:
# downloading, uncompressing, and saving EPA FRS facility location data
url = "https://edg.epa.gov/data/public/OEI/FRS/FRS_Interests_Download.zip"
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    zip_ref.extractall("FRS_Interests_Download")

In [ ]:
# creating a feature from the FRS TRI data
arcpy.conversion.ExportFeatures(
    in_features= cwd + "\\FRS_Interests_Download\\FRS_INTERESTS.gdb\\TRI",
    out_features="FRS"
)

In [ ]:
# joining tri data with frs lat/lons for spatial plotting
arcpy.management.JoinField(
    in_data="FRS",
    in_field="REGISTRY_ID",
    join_table="TRI_ET",
    join_field="FRS_ID_str",
    fields="STANDARD_PARENT_CO_NAME;INDUSTRY_SECTOR;ON_SITE_RELEASE_TOTAL",
    fm_option="NOT_USE_FM",
    field_mapping=None,
    index_join_fields="NO_INDEXES"
)

# exporting joined data as a feature
arcpy.conversion.ExportFeatures(
    in_features="FRS",
    out_features="TRI_XY",
    where_clause="ON_SITE_RELEASE_TOTAL IS NOT NULL",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping=None, 
    sort_field=None
)

## Air pollution data (RSEI)
The EPA also publishes the [Risk-Screening Environmental Indicator (RSEI)](https://www.epa.gov/rsei) model, which is a raster dataset showing modeled pollution exposure across the entire country. I used the 2022 census tract-level version of the RSEI data (the most recent available RSEI dataset in this format) so that it could be compared directly to census data.

In [ ]:
# downloading, uncompressing, and saving rsei data by census tract
url = "http://abt-rsei.s3.amazonaws.com/microdata2022/census_agg/censusmicrotract2022_2022_aggregated.csv.gz"
rsei_cens = pd.read_csv(url, compression="gzip")
rsei_cens["ToxConcP"] = rsei_cens.ToxConc.rank(pct=True)
rsei_cens.to_csv("rsei_cens.csv")

In [ ]:
# downloading, uncompressing, and census tract border data
url = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_tract_500k.zip"
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    zip_ref.extractall('cb_2020_us_tract_500k')

In [ ]:
# importing rsei data as csv, exporting as table (since it has no associated spatial information yet)
arcpy.conversion.ExportTable(
    in_table="rsei_cens.csv",
    out_table="rseicens_ET",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping=None,
    sort_field=None
)

# cleaning GEOID field for joining with census tract shapefiles
arcpy.management.CalculateField(
    in_table="rseicens_ET",
    field="GeoIDstr",
    expression='"!GeoID!".zfill(11)',
    expression_type="PYTHON3",
    code_block="",
    field_type="TEXT",
    enforce_domains="NO_ENFORCE_DOMAINS"
)

# importing census tract shapefile
arcpy.management.MakeFeatureLayer(
    in_features="cb_2020_us_tract_500k/cb_2020_us_tract_500k.shp", 
    out_layer="cens_tracts"
)
arcpy.management.DefineProjection(
    in_dataset="cens_tracts",
    coor_system=open("cb_2020_us_tract_500k/cb_2020_us_tract_500k.prj").read()
)

# joining rsei data with census tracts for spatial plotting
arcpy.management.AddJoin(
    in_layer_or_view="cens_tracts",
    in_field="GEOID",
    join_table="rseicens_ET",
    join_field="GeoIDstr",
    join_type="KEEP_ALL",
    index_join_fields="NO_INDEX_JOIN_FIELDS",
    rebuild_index="NO_REBUILD_INDEX",
    join_operation=""
)

# exporting joined data (now with associated spatial information) as a feature
arcpy.conversion.ExportFeatures(
    in_features="cens_tracts",
    out_features="rsei_cens",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping=None,
    sort_field=None
)

# creating raster layer of rsei data for visualization
arcpy.conversion.PolygonToRaster(
    in_features="rsei_cens",
    value_field="ToxConc",
    out_rasterdataset="rsei_cens_raster",
    cell_assignment="CELL_CENTER",
    priority_field="NONE",
    cellsize=0.01,
    build_rat="BUILD"
)

# clearing memory
arcpy.Delete_management("rseicens_ET")
arcpy.Delete_management("cens_tracts")

## Census demographic data
The census publishes yearly [American Community Survey](https://www.census.gov/programs-surveys/acs.html) data on a wide range of variables; I pulled the demographic data I was interested in using the `tidycensus` R package and am importing it into ArcGIS here.

In [ ]:
# importing from shapefile pulled from the census API in R
arcpy.management.MakeFeatureLayer(
    in_features="cens_demg_R/cens_demg_R.shp", 
    out_layer="cens_demg"
)
arcpy.management.DefineProjection(
    in_dataset="cens_demg",
    coor_system=open("cens_demg_R/cens_demg_R.prj").read()
)

# exporting joined data as a feature
arcpy.conversion.ExportFeatures(
    in_features="cens_demg",
    out_features="cens_demg",
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    field_mapping=None, 
    sort_field=None
)

# clearing memory
arcpy.Delete_management("cens_demg")

## HOLC grading data
In the 1930s, the Home Owners’ Loan Corporation (HOLC) created maps grading neighborhoods on desirability based on income, race, and other factors. These maps were historically used to enforce segregation.

In [ ]:
# pulling redlining data from ArcGIS Online
url = "https://services.arcgis.com/ak2bo87wLfUpMrt1/ArcGIS/rest/services/MappingInequalityRedliningAreas_231211/FeatureServer/0"

# creating a feature
arcpy.conversion.ExportFeatures(
    in_features=url,
    out_features="HOLC"
)